# WO-07 — WO-06 adjustments, class words, and the nearest river
Part A (so far): the ring sentences state only extremes, "all" and "all but one"; S6.4 is gone; S6.5 is widened; population density and human footprint are left out of the sentences (kept in the fill notes). Wording is in `profile_templates_v6_target.md` (the build target until the swap); the rules are in `scripts/edop/whg/wo06_rings.py`. Runs on the ring table saved in WO-06 (`output/edop/whg/wo06_ring_values.csv`); nothing is re-fetched. Read-only except `wo07_filled_profiles.md` and `output/edop/whg/wo07_fill.csv`.

In [1]:
%matplotlib inline
# Cell 1 — Setup: wording file (the v6 target), server check, saved ring, flow and river tables
import sys, time, warnings
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'edop').exists())
sys.path.insert(0, str(ROOT))
from scripts.shared.db_utils import db_connect
from scripts.edop.whg import wo03_common as C
from scripts.edop.whg import wo04_fill as W
from scripts.edop.whg import wo06_rings as R
from scripts.edop.whg import wo07_rivers as RV

W.TEMPLATES_PATH = ROOT / 'docs' / 'design' / 'dashboard' / 'profile_templates_v6_target.md'   # the build target until the swap
conn = db_connect()
PAIRS = pd.read_csv(C.OUT / 'q1_pairs.csv', dtype={'tag': str})
FLOW = pd.read_csv(C.OUT / 'q2_flow_pairs.csv', dtype={'tag': str})
SAMPLE = C.population_points()
RING = R.load_table()
RIV = RV.load_table() if RV.RIVERS_CSV.exists() else None     # filled by Cell 5; Cell 2 uses it once it exists
print('commit', C.commit(), '| server:', W.get_json('/api/health'), '| wording:', W.templates_stamp(), '| ring table:', len(RING), 'basins | river table:', len(RIV) if RIV else 'not built yet', '| test places:', len(C.PLACES), '| sample points:', len(SAMPLE))

commit 883d9e8 | server: {'status': 'ok'} | wording: profile_templates_v6_target.md v6 (last commit c98901f; edited since: no); script wo04_fill.py (last commit c98901f; edited since: no); script wo06_rings.py (last commit 8ff50dc; edited since: no); script wo07_rivers.py (last commit 883d9e8; edited since: no) | ring table: 315 basins | river table: 574 | test places: 10 | sample points: 306


In [3]:
# Cell 2 — The ten test places: the filled_profiles (written to docs/design/dashboard/wo07_filled_profiles.md) and their ring and river sentences
OUTMD = ROOT / 'docs' / 'design' / 'dashboard' / 'wo07_filled_profiles.md'
t0 = time.perf_counter()
TEST = {}
parts = ['# WO-07: the dummy filled from real data, ten test places (filled_profiles)', '',
         'Generated by `notebooks/edop/whg/wo07_adjust_classes_rivers.ipynb` (commit ' + C.commit() + '). Every `yes` unit of `modal_dummy_annotated.md` is filled from data; each `maybe` or `no` unit is a placeholder in the form `⟦id maybe: needs ...⟧`. Ring comparison units (S2.3, S5.2, S6.2, S6.3, S6.5) and the much-larger-river unit (S3.3) are silent where they have nothing true to say. Wording is the template file\'s, not polished.', '',
         'Wording from `' + W.templates_stamp() + '`.', '']
L = []
for name, (lat, lon) in C.PLACES.items():
    F = W.build_facts(lat, lon, conn, PAIRS, FLOW, ring=RING, rivers=RIV)
    if F['pub'] is None:
        L.append(f'{name}: no basin'); continue
    U = W.fill_units(F)
    TEST[name] = (F, U)
    parts.append(W.render_filled_profile(name, F, U)); parts.append('---'); parts.append('')
    L.append(f'{name} ({F["n_nb"]} neighbours; {U.get("S6.ring", {}).get("note", "")})')
    for u in ('S2.3', 'S5.2', 'S6.2', 'S6.3', 'S6.5', 'S3.1', 'S3.3'):
        if u in U:
            L.append(f'   {u}: ' + (U[u]['text'] if U[u]['ok'] else f'[silent: {U[u]["why"]}]'))
OUTMD.write_text('\n'.join(parts))
print(f'wrote {OUTMD.name} | {W.templates_stamp()} | {time.perf_counter() - t0:.0f}s\n\n' + '\n'.join(L))

wrote wo07_filled_profiles.md | profile_templates_v6_target.md v6 (last commit c98901f; edited since: no); script wo04_fill.py (last commit c98901f; edited since: no); script wo06_rings.py (last commit 8ff50dc; edited since: no); script wo07_rivers.py (last commit 883d9e8; edited since: no) | 15s

Timbuktu (5 neighbours; 5 neighbours; areas km2 core 3,826, neighbours 921 to 24,966; the core is neither the smallest nor the largest by area; left out of the sentences: population density highest, human footprint more than 4 of 5)
   S2.3: Its basin receives less precipitation than three of the five basins around it.
   S5.2: Its basin has less pasture than three of the five basins around it.
   S6.2: Two of them drain into it. It drains into one of them.
   S6.3: [silent: no measure on which all, or all but one of, the 2 basins across its divides lie on one side]
   S6.5: Of the six basins that meet here, it has the lowest mean elevation, the least relief and the gentlest slopes.
   S3.1: 

In [ ]:
# Cell 3 — The sample: 306 basin interior points through the ring rules; saved to a CSV (about 8 min)
rows, t_start = [], time.perf_counter()
for k, s in enumerate(SAMPLE, 1):
    F = W.build_facts(float(s['lat']), float(s['lon']), conn, PAIRS, FLOW, ring=RING)
    U = W.fill_units(F)
    d = dict(core_id=int(s['hybas_id_l6']), zone=s['climate_zone'], n_nb=F['n_nb'])
    for u in ('S2.3', 'S5.2', 'S6.2', 'S6.3', 'S6.5'):
        d[u] = int(U[u]['ok']); d[u + '_na'] = int(U[u].get('na', False)); d[u + '_why'] = U[u]['why'] or ''
    tr = {u: U[u].get('trace', {}) for u in ('S2.3', 'S5.2', 'S6.3', 'S6.5', 'S6.ring')}
    for u in ('S2.3', 'S5.2'):
        d[u + '_kind'], d[u + '_ext'], d[u + '_used'] = tr[u].get('kind', ''), len(tr[u].get('ext', [])), len(tr[u].get('used', []))
    d['S6.3_across'], d['S6.3_all'], d['S6.3_abo'] = tr['S6.3'].get('across', 0), len(tr['S6.3'].get('hit', [])), len(tr['S6.3'].get('abo', []))
    d['S6.3_first'] = (tr['S6.3'].get('hit') or tr['S6.3'].get('abo') or [('', '')])[0][0]
    d['S6.5_ext'], d['S6.5_used'] = len(tr['S6.5'].get('ext', [])), len(tr['S6.5'].get('used', []))
    d['S6.5_first'] = ' '.join(f'{a}' for a, b in tr['S6.5'].get('used', []))
    d['core_smallest'], d['core_largest'] = tr['S6.ring'].get('smallest'), tr['S6.ring'].get('largest')
    rows.append(d)
    if k % 50 == 0:
        print(f'  {k} of {len(SAMPLE)} in {time.perf_counter() - t_start:.0f}s')
S5 = pd.DataFrame(rows)
S5.to_csv(C.OUT / 'wo07_fill.csv', index=False)
print(f'rows saved: {len(S5)} | this run took {time.perf_counter() - t_start:.0f}s')

In [ ]:
# Cell 4 — Firing rates over the sample: how often each unit speaks, what kind of sentence, how often the cap bites
L = [f'sample basins: {len(S5)}']
for u in ('S2.3', 'S5.2', 'S6.2', 'S6.3', 'S6.5'):
    L.append(f'{u}: speaks {int(S5[u].sum())} | silent (nothing true to say) {int(S5[u + "_na"].sum())} | could not fill {int(((S5[u] == 0) & (S5[u + "_na"] == 0)).sum())}')
L.append('')
for u, cap in (('S2.3', R.CAP['S2_3']), ('S5.2', R.CAP['S5_2'])):
    L.append(f'{u}: kind {S5[u + "_kind"].value_counts().to_dict()} | extremes found per basin {S5[u + "_ext"].value_counts().sort_index().to_dict()} | cap {cap} bites at {int((S5[u + "_ext"] > cap).sum())}')
L.append(f'S6.5: extremes found per basin {S5["S6.5_ext"].value_counts().sort_index().to_dict()} | cap {R.CAP["S6_5"]} bites at {int((S5["S6.5_ext"] > R.CAP["S6_5"]).sum())} of {int((S5["S6.5"] == 1).sum())} that speak')
spoke = S5[S5['S6.3'] == 1]
L.append(f'S6.3: speaks at {len(spoke)} ("all" {int((spoke["S6.3_all"] > 0).sum())}, "all but one" {int(((spoke["S6.3_all"] == 0) & (spoke["S6.3_abo"] > 0)).sum())}) | first measure: ' + str(spoke['S6.3_first'].value_counts().to_dict()))
sp = S5[S5['S6.5'] == 1]
L.append('S6.5 measure named first: ' + str(sp['S6.5_first'].str.split().str[0].value_counts().to_dict()))
L.append('')
S5['nb_band'] = pd.cut(S5['n_nb'], [-1, 2, 4, 6, 8, 99], labels=['0-2', '3-4', '5-6', '7-8', '9+'])
S5['any_sentence'] = ((S5['S2.3'] + S5['S5.2'] + S5['S6.3'] + S5['S6.5']) > 0).astype(int)
S5['n_sentences'] = S5['S2.3'] + S5['S5.2'] + S5['S6.3'] + S5['S6.5']
g = S5.groupby('nb_band', observed=True)[['S2.3', 'S5.2', 'S6.3', 'S6.5', 'any_sentence']].mean()
g.insert(0, 'basins', S5.groupby('nb_band', observed=True).size())
L.append('by number of neighbours (share of basins that speak):'); L.append(g.round(2).to_string())
L.append('ring sentences per basin (S2.3, S5.2, S6.3, S6.5; the flow sentence S6.2 not counted): ' + str(S5['n_sentences'].value_counts().sort_index().to_dict()))
print('\n'.join(L))

In [ ]:
# Cell 5 — Part C: the river table for the ten places, the 306 sample basins and the 258 World Heritage Cities (every cutoff tried is kept; resumable; a failing point is logged and skipped)
CITIES = pd.DataFrame(conn.execute("SELECT id, city, country, ST_Y(geom::geometry), ST_X(geom::geometry) FROM gaz.wh_cities ORDER BY id").fetchall(), columns=['id', 'city', 'country', 'lat', 'lon'])
points = ([(n, 'test', la, lo) for n, (la, lo) in C.PLACES.items()]
          + [(f"sample {int(s['hybas_id_l6'])}", 'sample', float(s['lat']), float(s['lon'])) for s in SAMPLE]
          + [(f"{r.city}, {r.country}", 'wh_cities', float(r.lat), float(r.lon)) for r in CITIES.itertuples()])
t0 = time.perf_counter()
n_rows, n_new, failed = RV.fetch_table(points, conn)
RIV = RV.load_table()
print(f'river table: {n_rows} points in {RV.RIVERS_CSV.name} | {n_new} fetched this run | {len(failed)} failed | {time.perf_counter() - t0:.0f}s')
for lab, err in failed[:20]:
    print('  FAILED:', lab, '|', err)

In [ ]:
# Cell 6 — S3.1 in three tiers: a named river of class 4 or larger (100 m3/s and up) that is close, an unnamed river of class 5 or larger that is truly close, or silence
T = pd.DataFrame(list(RIV.values()))
rows = T.to_dict('records')
for r in rows:
    r['nm4'] = RV._name(r, 'r4')
    r['n4'] = r['r4_km'] if pd.notna(r['r4_km']) and r['nm4'] else np.nan      # distance to a named class 4 or larger river
T2 = pd.DataFrame(rows)
L4 = RV.NAMED_LIMITS['r4']
L = [f'current settings: a named river only if class 4 or larger, within {L4:g} km ("on" within {RV.ON_KM:g}); an unnamed river of class 5 or larger within {RV.NEAR_KM:g} km; otherwise silent']
for grp in ('sample', 'wh_cities'):
    d = T2[(T2['group'] == grp) & T2['hid'].notna()]
    n = len(d)
    L.append('')
    L.append(f'== {grp}: {n} points with a basin ==')
    L.append('tier 1, a named class 4 or larger river within D km: ' + ' | '.join(f'{D:g} km {int((d["n4"] <= D).sum())} ({100 * (d["n4"] <= D).sum() / n:.0f}%)' for D in (1.5, 5, 10, 15, 25, 40)))
    t1 = d['n4'] <= L4
    L.append('tier 2, an unnamed river of class 5 or larger within D km, among those not in tier 1: ' + ' | '.join(f'{D:g} km {int((~t1 & (d["r5_km"] <= D)).sum())} ({100 * (~t1 & (d["r5_km"] <= D)).sum() / n:.0f}%)' for D in (1.5, 3, 5, 10)))
    t2 = ~t1 & (d['r5_km'] <= RV.NEAR_KM)
    L.append(f'current settings: tier 1 {int(t1.sum())} ({100 * t1.sum() / n:.0f}%) | tier 2 {int(t2.sum())} ({100 * t2.sum() / n:.0f}%) | silent {int((~t1 & ~t2).sum())} ({100 * (~t1 & ~t2).sum() / n:.0f}%)')
c = T2[(T2['group'] == 'wh_cities') & T2['hid'].notna()].copy()
L.append('')
L.append(f'World Heritage Cities in tier 1 (a named class 4 or larger river within {L4:g} km), nearest first:')
L.append(c[c['n4'] <= L4].sort_values('n4')[['label', 'nm4', 'n4']].round(1).to_string(index=False, header=False))
L.append('')
L.append('the ten places:')
for n_, (la, lo) in C.PLACES.items():
    r = T2[T2['key'] == RV.key(la, lo)]
    if r.empty:
        L.append(f'   {n_}: not in the table'); continue
    r = r.iloc[0]
    L.append(f"   {n_}: class 4+ {r['r4_km']:.1f} km" + (f" ({r['nm4']})" if r['nm4'] else ' (no name)') + f" | class 5+ {r['r5_km']:.1f} km")
print('\n'.join(L))